# ai

> Ask a language model which sentences have a subject that does not do what its verb says

In [ ]:
#| default_exp ai

In [ ]:
#| hide
from nbdev.showdoc import *

NB: this file is designed for an LLM reader, not a human. Add it to your LLM's context and ask it to explain.

- `ai_texts`, `ai_paths`: flag sentences like "Rows carry their source": a model compares each with two example groups    /no rule finds these reliably
- findings: candidates for a person or stronger model; outside the rules' score
- needs `pip install slopometer[ai]`, `GEMINI_API_KEY`; ~6 cents per 4,000 words by default


In [ ]:
#| export
import itertools, random
from importlib.resources import files
from fastcore.utils import *
from fastcore.meta import delegates
from fastcore.parallel import parallel_async
from slopometer.core import *
from slopometer.segment import *
from slopometer.segment import _item
from slopometer.syntax import *
from slopometer.score import *
from slopometer.score import _read_file, _doc_paths

In [ ]:
from fastcore.test import *

## The example groups

`judge_groups.txt` ships inside the package. Group A holds sentences that are not literally true as written: a number "catches" payments, silence "signs" an agreement, settings "load". Group B holds sentences that state a fact in the normal words for it, including instructions and passives. The two groups cover a range of subjects and lengths. Some group B sentences are unrelated to the rest, such as a greeting. We add a sentence to group A when the check misses it in a real document, and we add to group B or remove an example when the check flags a sentence wrongly.

In [ ]:
#| export
def groups():
    "The group A and group B example sentences that ship with the package"
    a,b = files('slopometer').joinpath('judge_groups.txt').read_text().split('# Grammar group B')
    f = lambda t: [l.strip() for l in t.splitlines() if l.strip() and not l.startswith('#')]
    return f(a),f(b)

In [ ]:
ga,gb = groups()
assert ga and gb and not any(s.startswith('#') for s in ga+gb)
ga[:3], gb[:3]

## Sentences

The model judges sentences, so `sentences` splits a document into them. It reads the paragraphs and list items that `segment` returns. Headings are left out, and `segment` has already dropped fenced code, tables and block quotes. spaCy finds the sentence boundaries in each line of the scrubbed text, where inline code cannot confuse it. Each `Sent` holds the sentence on one line as a reader sees it. Inline code stays. Link targets and emphasis markers are removed, since scrubbing turned them into spaces. It also keeps its offsets in the document and the index of its paragraph.

Fragments of fewer than `min_words` words are skipped. Most are list items such as "fast startup", which have no verb to judge.

In [ ]:
#| export
class Sent:
    def __init__(self,
        start, # Character offset of the sentence in the document
        end, # Character offset one past its end
        text, # The sentence on one line, with inline code kept and other markup removed
        para, # Index of its paragraph or list item among the judged blocks
    ): store_attr()
    def __repr__(self): return f'<{self.para} @{self.start}: {self.text!r}>'

_tags = re.compile(r'^[ \t]*(?:</?[A-Za-z][\w:.-]*>[ \t]*)+$', re.M)
_stop = re.compile(r'''[.?!]['")\]*_]*$''')
def _quoted(s, i):
    "Is position `i` of `s` inside a double-quoted passage that closes later?"
    return (s.count('"', 0, i)%2 == 1 and '"' in s[i:]) or (s.count('“', 0, i) > s.count('”', 0, i) and '”' in s[i:])

_seen = re.compile(r"""<https?://[^\s>]+>|(?<!\]\()https?://[^\s<>\[\]]*[^\s<>\[\]().,;:!?'"]|(?<![\w`])__[a-z][a-z0-9_]*?__(?![\w`])""")
def _keep(o, s):
    "`s`, the scrubbed `o`, with the URLs and dunder names that a reader of `o` sees copied back from it"
    for m in _seen.finditer(o): s = s[:m.start()] + m[0] + s[m.end():]
    return s
_abbr = re.compile(r'(?<![\w.])(?:incl|excl|approx|cf|esp|resp|viz|ca|fig|figs|eq|eqs|sec|vol|al|misc|est|avg|std|ref|refs|ex|e\.g|i\.e|etc|vs|no|nos|pp)\.$', re.I)

def sentences(txt, min_words=3):
    "Sentences of `min_words`+ words in the paragraphs and list items of markdown `txt`; a newline ends a sentence"
    blocks = [b for b in segment(txt) if b.kind != 'heading']
    def clean(b):
        s = _tags.sub(lambda m: ' '*len(m[0]), _keep(b.txt, scrub(b.txt)))
        m = _item.match(s) if b.kind == 'item' else None
        return ' '*m.end() + s[m.end():] if m else s
    lines = [(k, b, m.start(), m[0]) for k,b in enumerate(blocks) for m in re.finditer(r'[^\n]+', clean(b))]
    res = []
    for (k,b,off,_),d in zip(lines, get_nlp().pipe(l[3] for l in lines)):
        spans = []
        for s in d.sents:
            toks = [t for t in s if not t.is_space]
            if not toks: continue
            a,e = toks[0].idx, toks[-1].idx+len(toks[-1])
            prev = d.text[spans[-1][0]:spans[-1][1]] if spans else ''
            if spans and (spans[-1][1] == a or _quoted(d.text, a) or not _stop.search(prev) or (_abbr.search(prev) and not d.text[a].isupper())): spans[-1][1] = e
            else: spans.append([a, e])
        for a,e in spans:
            ws = ''.join(o for o,c in zip(b.txt[off+a:off+e], d.text[a:e]) if c != ' ' or o.isspace()).split()
            if len(ws) >= min_words: res.append(Sent(b.start+off+a, b.start+off+e, ' '.join(ws), k))
    return res

This document has a heading, a wrapped paragraph with inline code, a fenced block, and a list:

In [ ]:
DOC = """# widget

Rows carry their source. Call `enhance()` to start.
fork    exact clone of this session

```python
rows.carry(source)  # code is never judged
```

- [Each event](events.md) knows its *timestamp*.
- fast startup
"""
ss = sentences(DOC)
ss

The offsets locate each sentence in the document. A newline always ends a sentence. The third line of `DOC` has no full stop and is one sentence. The list marker is not part of the text:

In [ ]:
test_eq([s.text for s in ss], ['Rows carry their source.', 'Call `enhance()` to start.', 'fork exact clone of this session', 'Each event knows its timestamp.'])
test_eq(DOC[ss[2].start:ss[2].end], 'fork    exact clone of this session')
test_eq([s.para for s in ss], [0, 0, 0, 1])


Some prompt files wrap their sections in tags such as `<rules>`. `sentences` drops a tag that has a line to itself, and keeps other text in angle brackets, such as a placeholder. It also rejoins a sentence that spaCy splits in the wrong place. One such place is a full stop with no space after it, as in `Answer.AI`. Another is a place with no full stop at all, such as before "so" in the second sentence here:

In [ ]:
tagged = "<rules>\nWe work without an Answer.AI account. Asset URLs carry a content hash (`vsrc`) so browsers can cache them.\nCheck each notebook with nbdev-test <nb>\n</rules>\n"
test_eq([s.text for s in sentences(tagged)], ['We work without an Answer.AI account.', 'Asset URLs carry a content hash (`vsrc`) so browsers can cache them.',
    'Check each notebook with nbdev-test <nb>'])

spaCy also splits some sentences that are whole. `sentences` rejoins three kinds. A colon does not end a sentence, so a label stays with what it introduces. A passage in double quotes stays in one piece, even when it holds a full stop. A full stop after a common abbreviation, such as "incl." or "cf.", does not end a sentence when the next word does not start with a capital letter:

In [ ]:
dense = """- run: glyphs, names, groups, no spaces between. runs: `+/÷≢`, `x.[1 2]`.
- Open with "NB: this file is for an LLM reader, not a human. Add it to your context." It tells a person why the file reads oddly.
- status: ask it to report, incl. long runs. MUST NOT brief it.
"""
test_eq([s.text for s in sentences(dense)], ['run: glyphs, names, groups, no spaces between.', 'runs: `+/÷≢`, `x.[1 2]`.',
    'Open with "NB: this file is for an LLM reader, not a human. Add it to your context."', 'It tells a person why the file reads oddly.',
    'status: ask it to report, incl. long runs.', 'MUST NOT brief it.'])

`scrub` blanks some text that a reader sees, as if it were markup: a bare URL or autolink, and a name such as `__call__`, whose underscores look like emphasis. `sentences` copies these back from the source before it splits, so a sentence keeps them:

In [ ]:
seen = "Docs: https://fastcore.fast.ai/tools.html.md and https://fastcore.fast.ai/nbio.html.md\nCall the op object, never type(op).__call__ directly.\n"
test_eq([s.text for s in sentences(seen)], ['Docs: https://fastcore.fast.ai/tools.html.md and https://fastcore.fast.ai/nbio.html.md',
    'Call the op object, never type(op).__call__ directly.'])

A long document goes to the model in chunks of about 1,000 words, the size the prompt was tested on. `_chunks` never divides a paragraph. Sentence numbers run through the whole document.

In [ ]:
#| export
def _chunks(sents, max_words=1000):
    "Runs of whole paragraphs of about `max_words` words, as lists of (number, `Sent`) pairs"
    res,cur,n = [],[],0
    for _,g in itertools.groupby(enumerate(sents), lambda o: o[1].para):
        g = list(g)
        w = sum(len(s.text.split()) for _,s in g)
        if cur and n+w > max_words: res,cur,n = res+[cur],[],0
        cur,n = cur+g,n+w
    return res+[cur] if cur else res

In [ ]:
cs = _chunks(ss, max_words=10)
test_eq([[i for i,_ in c] for c in cs], [[0, 1, 2], [3]])

## The prompt

`ai_prompt` gives the model one sentence of framing, the two groups, and the numbered sentences with a blank line between paragraphs. With a `seed` it shuffles the order of the examples in each group. It asks for a verdict on every line. A list of only the flagged numbers would be shorter, but in tests it found fewer of the sentences we wanted.

`SYSTEM` stops `gemini-3.8-flash` from thinking before it answers. Without it the model thinks in most calls, which costs several times more and finds no more.

In [ ]:
#| export
MODEL = 'gemini/models/gemini-3.8-flash'
SYSTEM = 'Answer immediately. Do not deliberate or plan before you answer.'

def ai_prompt(chunk, seed=None):
    "Prompt asking for `H` or `F` on each sentence of `chunk`, a list of (number, `Sent`) pairs; `seed` shuffles the examples"
    a,b = groups()
    if seed is not None:
        for g in (a,b): random.Random(seed).shuffle(g)
    lines,prev = [],None
    for i,s in chunk:
        if prev not in (None, s.para): lines.append('')
        lines.append(f'[{i}] {s.text}')
        prev = s.para
    nl = '\n'
    return f"""Some sentences make the reader stop and work out what really happened and who did it.

Hard sentences:

{nl.join(a)}

Fine sentences:

{nl.join(b)}

Judge each numbered sentence below in the same way. Answer with one line per sentence, in the form `12 H` for a hard sentence or `12 F` for a fine one, and nothing else.

{nl.join(lines)}"""

In [ ]:
print(ai_prompt(cs[0] + cs[1])[-330:])

`parse_verdicts` reads the reply. It accepts the number with or without brackets. It also accepts `[3] 3 H`, a form in which `gemini-3.8-flash` repeats the number:

In [ ]:
#| export
_verdict = re.compile(r'^\W*(\d+)\W+(?:\1\W+)?([HF])\b', re.M)
def parse_verdicts(txt):
    "Sentence number -> True for `H`, from a reply with one verdict per line"
    return {int(i): v == 'H' for i,v in _verdict.findall(txt)}

In [ ]:
test_eq(parse_verdicts('0 F\n[1] H\n2: H\n[3] 3 H'), {0: False, 1: True, 2: True, 3: True})

## Results

An `AIFinding` is one sentence that at least one run flagged. Its weight is the number of runs that flagged it. An `AIResult` keeps every sentence sent to the model in `sents`. `votes` holds the number of runs that flagged each one, and `judged` the number that gave any verdict on it. A failed call or an incomplete reply lowers `judged`. `missing` is the total of verdicts that never came back. The report shows it when it is above zero. The density is the percentage of the sentences flagged.

In [ ]:
#| export
class AIFinding(Finding):
    def __init__(self,
        start, # Character offset where the sentence starts
        end, # Character offset one past its end
        text, # The sentence
        votes, # Runs that flagged it
        judged, # Runs that gave a verdict on it
    ):
        super().__init__('ai', None, start, end, text, votes)
        store_attr('votes,judged')
    def __repr__(self): return f'[{self.votes}/{self.judged}] {self.text!r}'

class AIResult(Result):
    def __init__(self,
        txt, # The judged document
        words, # Word count across all scored blocks
        sents=(), # Every `Sent` sent to the model
        votes=(), # Runs that flagged each of `sents`
        judged=(), # Runs that gave a verdict on each of `sents`
        runs=5, # Runs made
    ):
        fs = [AIFinding(s.start, s.end, s.text, v, j) for s,v,j in zip(sents, votes, judged) if v]
        super().__init__(txt, sorted(fs, key=lambda f: (-f.votes, f.start)), words, min_words=0)
        store_attr('sents,votes,judged,runs')
    @property
    def missing(self):
        "Verdicts that failed calls and incomplete replies left out"
        return sum(self.runs-j for j in self.judged)
    def _header(self):
        res = f'{len(self.findings)} of {len(self.sents)} sentences flagged in any of {self.runs} runs ({self.density}%), worst {self.worst}'
        return f'{res}, {self.missing} verdicts missing' if self.missing else res
    def as_dict(self):
        "JSON-ready record of every sentence sent to the model, with its votes and source location"
        return dict(path=self.path and str(self.path), words=self.words, runs=self.runs, missing=self.missing,
            sentences=[dict(n=i, text=s.text, para=s.para, votes=v, judged=j, **self.location(s))
                for i,(s,v,j) in enumerate(zip(self.sents, self.votes, self.judged))])

@patch(as_prop=True)
def density(self:AIResult):
    "Percentage of the sentences that any run flagged"
    return round(100*len(self.findings)/max(len(self.sents), 1), 1)

Two of five runs flagged the first sentence here. One run gave no verdict on the last:

In [ ]:
r = AIResult(DOC, scored_words(segment(DOC)), ss, [2, 0, 0, 0], [5, 5, 5, 4])
test_eq((len(r.findings), r.density, r.worst, r.missing), (1, 25.0, 2, 1))
test_eq(r.as_dict()['sentences'][3]['line'], 10)
r

## Judging texts and files

Each call returns the verdicts for one chunk, or the exception if the call failed. `_count` adds them up for each sentence, reading only the chunk's own sentence numbers from a reply.

In [ ]:
#| export
def _count(jobs, outs, ns):
    "Runs that flagged each sentence, and runs that gave a verdict on it, for documents of `ns` sentences"
    votes,judged = ([[0]*n for n in ns] for _ in range(2))
    for (k,c,_),vs in zip(jobs, outs):
        if isinstance(vs, Exception): continue
        for i,_ in c:
            if i not in vs: continue
            judged[k][i] += 1
            votes[k][i] += vs[i]
    return votes,judged

Here three runs judge one chunk of two sentences. The first reply covers both, the second leaves one out, and the third call fails:

In [ ]:
jobs = [(0, list(enumerate(ss[:2])), run) for run in range(3)]
outs = [{0: True, 1: False}, {0: True}, TimeoutError()]
test_eq(_count(jobs, outs, [2]), ([[2, 0]], [[2, 1]]))

`ai_texts` judges several documents at once. Every chunk of every document is sent `runs` times, and all of those calls share one limit of `n_workers` at a time. Each run shuffles the examples with its run number as the seed. With thinking off, runs of one identical prompt give nearly identical answers, and repeating them finds nothing new. A sentence is flagged when any run flags it. A document with no sentence to judge gets no request.

A call that fails does not stop the others. Each result reports how many verdicts are missing. `ai_texts` raises an error only when no verdict came back at all, since that run says nothing about any document.

In [ ]:
#| export
async def _ask(job, model):
    "Verdicts on a chunk for one run, from `job`, a (chunk, run number) pair; the exception if the call fails"
    chunk,run = job
    from fastllm.acomplete import acomplete
    from fastllm.chat import contents
    from aidialog.msg_parts import mk_msgs
    try: c = await acomplete(mk_msgs([ai_prompt(chunk, run)]), model, system=SYSTEM, reasoning_effort='low')
    except Exception as e: return e
    return parse_verdicts(contents(c).text)

async def ai_texts(
    txts, # Markdown documents
    runs=5, # Times to send each chunk
    model=MODEL, # `fastllm` model path
    n_workers=16, # Model calls in flight at once, across all documents
):
    "One `AIResult` per document in `txts`, flagging each sentence that any run calls hard"
    docs = [(t, scored_words(segment(t)), sentences(t)) for t in txts]
    jobs = [(k, c, r) for k,(_,_,ss) in enumerate(docs) for c in _chunks(ss) for r in range(runs)]
    outs = await parallel_async(_ask, [(c, r) for _,c,r in jobs], model, n_workers=n_workers)
    votes,judged = _count(jobs, outs, [len(ss) for _,_,ss in docs])
    if jobs and not any(map(any, judged)): raise first(outs, risinstance(Exception)) or RuntimeError(f'{model} gave no verdicts')
    return [AIResult(t, w, ss, v, j, runs) for (t,w,ss),v,j in zip(docs, votes, judged)]

@delegates(ai_texts)
async def ai_paths(paths, **kwargs):
    "One `AIResult` per file in `paths`, which takes what `score_paths` takes, with exhash addresses"
    read = [_read_file(p) for p in _doc_paths(paths)]
    rs = await ai_texts([t for t,_ in read], **kwargs)
    for r,(_,loc) in zip(rs, read): vars(r).update(loc)
    return rs

A document with no sentence to judge gets no request and no findings:

In [ ]:
empty = (await ai_texts(['# Only a heading\n\n- fast startup\n']))[0]
test_eq((empty.sents, empty.findings, empty.density), ([], [], 0.0))

With `GEMINI_API_KEY` set, judge a file or a directory. `--ai` on the command line does the same:

In [ ]:
#| eval: false
rs = await ai_paths('08_corpus.ipynb')
rs[0]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()